# Import and Setup

In [ ]:
from IPython.display import display, Markdown

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
import keys
NRP_TOK = keys.NRP_TOK

nrp_llm_url = "https://ellm.nrp-nautilus.io/v1"

model = ChatOpenAI(model = 'gpt-oss',
                   api_key = NRP_TOK,
                   base_url = nrp_llm_url,
                   use_responses_api=False)

# Online Travel Agent

In [ ]:
client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

In [ ]:
tools = await client.get_tools()

In [ ]:
tools

In [ ]:
for i in tools:
    print(i.name)
    print(i.description)

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [ ]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st, 2026")]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
response

In [ ]:
print(response["messages"][-1].content)

# Local shell

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "python",
            "args": ["mcp_shelldemo.py"]
        }
    }
)

In [ ]:
tools = await client.get_tools()

In [ ]:
tools

In [ ]:
# will not get anything currentnly
resources = await client.get_resources("time")

In [ ]:
resources

In [ ]:
# will not get anything currently
prompt = await client.get_prompt("time", "prompt")
prompt = prompt[0].content

In [ ]:
time_agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You tell time."
)

In [ ]:
config = {"configurable": {"thread_id": "2"}}

response = await agent.ainvoke(
    {"messages": [SystemMessage(content='Respond in plain text.'),
                  HumanMessage(content="Tell me the current time")]},
    config=config
)

In [ ]:
response

In [ ]:
response['messages'][-1].content

In [ ]:
print(response['messages'][-1].content)

In [ ]:
client = MultiServerMCPClient(
    {
        "langchain-ai": {
            "transport": "stdio",
            "command": "python",
            "args": ["mcp_itemdemo.py"]
        }
    }
)

In [ ]:
tools = await client.get_tools()

In [ ]:
tools

In [ ]:
resources = await client.get_resources()

In [ ]:
resources

In [ ]:
resources = await client.get_resources("langchain-ai")

In [ ]:
resources

In [ ]:
# Requires server_name and prompt_name
prompt = await client.get_prompt("langchain-ai", "langchain-ai prompt")
prompt = prompt[0].content

In [ ]:
prompt

In [ ]:
# NRP_TOK = keys.NRP_TOK
# nrp_llm_url = "https://ellm.nrp-nautilus.io/v1"

# model = ChatOpenAI(model = 'gpt-oss',
#                  api_key = NRP_TOK,
#                  base_url = nrp_llm_url,
#                  # use_responses_api=False forces the classic chat.completions endpoint
#                  use_responses_api=False,
#                  temperature = 0.3)

agent = create_agent(model=model,
                     tools=tools)

In [ ]:
response = await agent.ainvoke(
    {"messages": [SystemMessage(content=prompt),
                  HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
)

In [ ]:
len(response['messages'])

In [ ]:
# response

In [ ]:
for i in response['messages']:
    try:
        print(i.tool_calls)
    except:
        print('No tool call')

In [ ]:
text = response['messages'][-1].content
display(Markdown(text))

# GitHub

In [ ]:
import githubtoken

In [ ]:
client = MultiServerMCPClient({
    "github": {
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {"Authorization": f"Bearer {githubtoken.GITHUB_TOKEN}"},
        "transport": "http"
    }
})

In [ ]:
tools = await client.get_tools()   

In [ ]:
for i in tools:
    print('-'*80)
    print(i.name)
    print('-'*80)
    print(i.description)
        

In [ ]:
resources = await client.get_resources() 

In [ ]:
resources

In [ ]:
tools[0]

In [ ]:
tools[0].args_schema

In [ ]:
response = await tools[0].ainvoke({'owner':'benjum',
                 'repo':'UCLAX-LLM',
                 'sha':'b4c1b4bba6f29abbf5895dc8d046de1999549bd1'})

In [ ]:
response

In [ ]:
import json

In [ ]:
json.loads(response[0]['text'])